In [2]:
# ==========================================
# Surface Crack Detection - One Folder Split
# ==========================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import numpy as np

# ----------------
# 1. Device
# ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ----------------
# 2. Paths
# ----------------
data_dir = '/Users/aparnabisht/Documents/ML/GPT_ML/SurfaceCrackDetection/imgdata/'   # change this to your folder path

# ----------------
# 3. Transforms
# ----------------
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ----------------
# 4. Load Full Dataset Twice
# ----------------
# We load twice so train and val can have different transforms
full_dataset_train = datasets.ImageFolder(data_dir, transform=train_transform)
full_dataset_val = datasets.ImageFolder(data_dir, transform=val_transform)

print("Classes:", full_dataset_train.classes)
print("Class to index:", full_dataset_train.class_to_idx)
print("Total images:", len(full_dataset_train))

# ----------------
# 5. Train/Val Split
# ----------------
targets = full_dataset_train.targets
indices = list(range(len(full_dataset_train)))

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2,
    stratify=targets,
    random_state=42
)

train_dataset = Subset(full_dataset_train, train_indices)
val_dataset = Subset(full_dataset_val, val_indices)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

# ----------------
# 6. DataLoaders
# ----------------
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# ----------------
# 7. Model
# ----------------
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

model = model.to(device)

# ----------------
# 8. Loss and Optimizer
# ----------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# ----------------
# 9. Training Function
# ----------------
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# ----------------
# 10. Validation Function
# ----------------
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# ----------------
# 11. Train Model
# ----------------
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")
    print("-" * 40)

# ----------------
# 12. Save Model
# ----------------
torch.save(model.state_dict(), "surface_crack_resnet18.pth")
print("\nModel saved!")

# ----------------
# 13. Confusion Matrix
# ----------------
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:")
print(cm)

# ----------------
# 14. Labeled Confusion Matrix
# ----------------
class_names = full_dataset_train.classes

print("\nConfusion Matrix with labels:\n")
print(f"{'':20}{'Pred: ' + class_names[0]:15}{'Pred: ' + class_names[1]:15}")
print(f"{'Actual: ' + class_names[0]:20}{cm[0][0]:15}{cm[0][1]:15}")
print(f"{'Actual: ' + class_names[1]:20}{cm[1][0]:15}{cm[1][1]:15}")

Using device: cpu
Classes: ['Negative', 'Positive']
Class to index: {'Negative': 0, 'Positive': 1}
Total images: 40000
Train size: 32000
Val size: 8000

Epoch 1/5
Train Loss: 0.1597
Train Accuracy: 0.9406
Validation Loss: 0.1763
Validation Accuracy: 0.9524
----------------------------------------

Epoch 2/5
Train Loss: 0.1231
Train Accuracy: 0.9543
Validation Loss: 0.1998
Validation Accuracy: 0.9144
----------------------------------------

Epoch 3/5
Train Loss: 0.1192
Train Accuracy: 0.9559
Validation Loss: 0.1680
Validation Accuracy: 0.9336
----------------------------------------

Epoch 4/5
Train Loss: 0.1154
Train Accuracy: 0.9584
Validation Loss: 0.1343
Validation Accuracy: 0.9575
----------------------------------------

Epoch 5/5
Train Loss: 0.1182
Train Accuracy: 0.9559
Validation Loss: 0.3497
Validation Accuracy: 0.8309
----------------------------------------

Model saved!

Confusion Matrix:
[[3999    1]
 [1352 2648]]

Confusion Matrix with labels:

                    Pred: 

In [3]:
# ==========================================
# Surface Crack Detection (FAST CPU VERSION)
# ==========================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import numpy as np

# ----------------
# 1. Device
# ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ----------------
# 2. Path
# ----------------
data_dir = '/Users/aparnabisht/Documents/ML/GPT_ML/SurfaceCrackDetection/imgdata/'    # change if needed

# ----------------
# 3. Transforms (smaller = faster)
# ----------------
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

# ----------------
# 4. Load Dataset Twice
# ----------------
full_dataset_train = datasets.ImageFolder(data_dir, transform=train_transform)
full_dataset_val = datasets.ImageFolder(data_dir, transform=val_transform)

print("Classes:", full_dataset_train.classes)
print("Class mapping:", full_dataset_train.class_to_idx)
print("Total samples:", len(full_dataset_train))

# ----------------
# 5. Train / Val Split
# ----------------
indices = list(range(len(full_dataset_train)))
targets = full_dataset_train.targets

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=targets,
    random_state=42
)

train_dataset = Subset(full_dataset_train, train_idx)
val_dataset = Subset(full_dataset_val, val_idx)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

# ----------------
# 6. DataLoaders (smaller batch = faster)
# ----------------
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# ----------------
# 7. Model (ResNet18 - frozen)
# ----------------
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

# ----------------
# 8. Loss & Optimizer
# ----------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# ----------------
# 9. Train Function
# ----------------
def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

# ----------------
# 10. Validation Function
# ----------------
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

# ----------------
# 11. Training Loop (short for speed)
# ----------------
num_epochs = 3

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc : {train_acc:.4f}")
    print(f"Val Loss  : {val_loss:.4f}")
    print(f"Val Acc   : {val_acc:.4f}")

# ----------------
# 12. Confusion Matrix
# ----------------
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:")
print(cm)

# ----------------
# 13. Pretty Print
# ----------------
class_names = full_dataset_train.classes

print("\nConfusion Matrix with labels:\n")
print(f"{'':20}{'Pred: ' + class_names[0]:15}{'Pred: ' + class_names[1]:15}")
print(f"{'Actual: ' + class_names[0]:20}{cm[0][0]:15}{cm[0][1]:15}")
print(f"{'Actual: ' + class_names[1]:20}{cm[1][0]:15}{cm[1][1]:15}")

# ----------------
# 14. Save Model
# ----------------
torch.save(model.state_dict(), "resnet18_fast_crack_model_baseline.pth")
print("\nModel saved!")

Using device: cpu
Classes: ['Negative', 'Positive']
Class mapping: {'Negative': 0, 'Positive': 1}
Total samples: 40000
Train size: 32000
Val size: 8000

Epoch 1/3
Train Loss: 0.0938
Train Acc : 0.9672
Val Loss  : 0.0280
Val Acc   : 0.9910

Epoch 2/3
Train Loss: 0.0666
Train Acc : 0.9758
Val Loss  : 0.0571
Val Acc   : 0.9786

Epoch 3/3
Train Loss: 0.0664
Train Acc : 0.9771
Val Loss  : 0.0242
Val Acc   : 0.9915

Confusion Matrix:
[[3979   21]
 [  47 3953]]

Confusion Matrix with labels:

                    Pred: Negative Pred: Positive 
Actual: Negative               3979             21
Actual: Positive                 47           3953

Model saved!


In [1]:
# ==========================================
# Surface Crack Detection - ResNet18 Baseline
# FAST CPU VERSION
# ==========================================

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import matplotlib.pyplot as plt

# ----------------
# 1. Device
# ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ----------------
# 2. Path
# ----------------
data_dir = "/Users/aparnabisht/Documents/ML/GPT_ML/SurfaceCrackDetection/imgdata/"

print("Current working directory:", os.getcwd())
print("Dataset exists:", os.path.exists(data_dir))

# ----------------
# 3. Transforms
# ResNet18 was pretrained using ImageNet normalization
# ----------------
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize,
])

val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    normalize,
])

# ----------------
# 4. Load Dataset
# ----------------
full_dataset_train = datasets.ImageFolder(data_dir, transform=train_transform)
full_dataset_val = datasets.ImageFolder(data_dir, transform=val_transform)

print("Classes:", full_dataset_train.classes)
print("Class mapping:", full_dataset_train.class_to_idx)
print("Total samples:", len(full_dataset_train))

# ----------------
# 5. Train / Val Split
# ----------------
indices = list(range(len(full_dataset_train)))
targets = full_dataset_train.targets

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=targets,
    random_state=42
)

train_dataset = Subset(full_dataset_train, train_idx)
val_dataset = Subset(full_dataset_val, val_idx)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

# ----------------
# 6. DataLoaders
# ----------------
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

# ----------------
# 7. Model: Frozen ResNet18
# ----------------
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

# ----------------
# 8. Loss & Optimizer
# ----------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# ----------------
# 9. Training Function
# ----------------
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

# ----------------
# 10. Evaluation Function
# ----------------
def evaluate(model, loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

# ----------------
# 11. Training Loop
# ----------------
num_epochs = 3

train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc : {train_acc:.4f}")
    print(f"Val Loss  : {val_loss:.4f}")
    print(f"Val Acc   : {val_acc:.4f}")

# ----------------
# 12. Plot Loss
# ----------------
plt.figure()
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

# ----------------
# 13. Plot Accuracy
# ----------------
plt.figure()
plt.plot(train_accs, label="Train Accuracy")
plt.plot(val_accs, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.show()

# ----------------
# 14. Confusion Matrix
# ----------------
all_preds = []
all_labels = []

model.eval()

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
class_names = full_dataset_train.classes

print("\nConfusion Matrix:")
print(cm)

print("\nConfusion Matrix with labels:\n")
print(f"{'':20}{'Pred: ' + class_names[0]:15}{'Pred: ' + class_names[1]:15}")
print(f"{'Actual: ' + class_names[0]:20}{cm[0][0]:15}{cm[0][1]:15}")
print(f"{'Actual: ' + class_names[1]:20}{cm[1][0]:15}{cm[1][1]:15}")

# ----------------
# 15. Classification Report
# ----------------
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

# ----------------
# 16. Save Model
# ----------------
model_path = "resnet18_surface_crack_fast_baseline.pth"
torch.save(model.state_dict(), model_path)

print(f"\nModel saved as: {model_path}")

/Users/aparnabisht/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Using device: cpu
Current working directory: /Users/aparnabisht/Documents/ML/GPT_ML/SurfaceCrackDetection
Dataset exists: True
Classes: ['Negative', 'Positive']
Class mapping: {'Negative': 0, 'Positive': 1}
Total samples: 40000
Train size: 32000
Val size: 8000


KeyboardInterrupt: 